In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/viagens_teste/bronze/arquivos/"
    )
)

In [0]:
df_a = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ";")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv("/Volumes/viagens_teste/bronze/arquivos/passagens/vianorte/*.csv")
)

In [0]:
df_a.printSchema()

df_a.columns

In [0]:
df_b = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ";")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv("/Volumes/viagens_teste/bronze/arquivos/passagens/atlas/*.csv")
)

In [0]:
df_b.printSchema()

df_b.columns

In [0]:
from pyspark.sql.functions import col, lit

In [0]:
df_a_padrao = df_a.select(
    col("Codigo").alias("id_solicitacao"),
    col("Nome Completo").alias("viajante"),
    col("Data Solicitacao").alias("data_solicitacao"),
    col("Data Ida").alias("data_viagem"),
    col("Centro de Custo").alias("centro_custo_origem"),
    col("Origem").alias("origem"),
    col("Destino").alias("destino"),
    col("Valor Total").alias("valor"),
    col("Situacao").alias("situacao"),
    col("Justificativa").alias("justificativa_original")
).withColumn(
    "agencia",
    lit("ViaNorte")
)

In [0]:
display(df_a_padrao)

In [0]:
df_b_padrao = df_b.select(
col("Nr Reserva").alias("id_solicitacao"),
col("Passageiro").alias("viajante"),
col("Criado Em").alias("data_solicitacao"),
col("Embarque").alias("data_viagem"),
col("CC").alias("centro_custo_origem"),
col("De").alias("origem"),
col("Para").alias("destino"),
col("Tarifa").alias("valor"),
col("Status").alias("situacao"),
col("Motivo Viagem").alias("justificativa_original")
).withColumn(
    "agencia",
    lit("Atlas Corporate")
)
    

In [0]:
display(df_b_padrao)

In [0]:
df_a_padrao.columns == df_b_padrao.columns

In [0]:
df_a_padrao = df_a_padrao.filter(
    col("id_solicitacao").isNotNull()
)

df_b_padrao = df_b_padrao.filter(
    col("id_solicitacao").isNotNull()
)

In [0]:
print("ViaNorte:", df_a_padrao.count())
print("Atlas:", df_b_padrao.count())

In [0]:
df_passagens = df_a_padrao.unionByName(df_b_padrao)

display(df_passagens)

### Limpeza das justificativas

Preservamos a justificativa original recebida da agência e criamos uma versão limpa para processamento textual, removendo quebras de linha e espaços excedentes.

In [0]:
from pyspark.sql.functions import trim, regexp_replace

df_passagens = df_passagens.withColumn(
    "justificativa_limpa",
    trim(
        regexp_replace(
            col("justificativa_original"),
            r"\s+",
            " "
        )
    )
)

In [0]:
df_passagens.select(
    "id_solicitacao",
    "agencia",
    "justificativa_original",
    "justificativa_limpa"
).show(10, truncate=False)

In [0]:
display(df_passagens)

In [0]:
df_a.count()

In [0]:
df_b_padrao.count()

In [0]:
print(df_a_padrao.count())
print(df_b_padrao.count())

In [0]:
print(df_passagens.count())

In [0]:
df_passagens.groupBy("agencia").count().show()

In [0]:
df_passagens.select(
    "agencia",
    "valor"
    ).show(20, truncate=False)

df_passagens.printSchema()

In [0]:
from pyspark.sql.functions import regexp_replace

In [0]:
df_passagens = df_passagens.withColumn(
    "valor_numerico",
    regexp_replace(
        regexp_replace(
            regexp_replace(
                col("valor"),
                "R\\$ ",
                ""
            ),
            "\\.",
            ""
        ),
        ",",
        "."
    ).cast("decimal(18,2)")
)

In [0]:
df_passagens.select(
    "agencia",
    "valor",
    "valor_numerico"
).show(20, truncate=False)

In [0]:
df_passagens.printSchema()

In [0]:
from pyspark.sql.functions import sum

df_passagens.select(
    sum("valor_numerico").alias("total_passagens")
).show()

In [0]:
df_passagens.groupBy("agencia").agg(
    sum("valor_numerico").alias("total_passagens")
).show()

In [0]:
from pyspark.sql.functions import datediff

In [0]:
df_passagens = df_passagens.withColumn(
    "dias_antecedencia",
    datediff(
        col("data_viagem"),
        col("data_solicitacao")
    )
)

In [0]:
df_passagens.select(
    "id_solicitacao",
    "data_solicitacao",
    "data_viagem",
    "dias_antecedencia"
).show(20, truncate=False)

In [0]:
from pyspark.sql.functions import when

In [0]:
df_passagens = df_passagens.withColumn(
    "prazo",
    when(
        col("dias_antecedencia") >= 10,
        "Dentro"
    ).otherwise("Fora")
)

In [0]:
df_passagens.select(
    "id_solicitacao",
    "data_solicitacao",
    "data_viagem",
    "dias_antecedencia",
    "prazo"
).show(20, truncate=False)

In [0]:
df_passagens.groupBy("prazo").count().show()

In [0]:
df_passagens.groupBy("prazo").agg(
    sum("valor_numerico").alias("valor_total")
).show()

In [0]:
df_passagens = df_passagens.withColumn(
    "tipo_despesa",
    lit("Aereo")
)

In [0]:
from pyspark.sql.functions import month

In [0]:
df_passagens = df_passagens.withColumn(
    "trimestre",
    when(
        month(col("data_viagem")).between(1, 3),
        "Primeiro Trimestre"
    )
    .when(
        month(col("data_viagem")).between(4, 6),
        "Segundo Trimestre"
    )
    .when(
        month(col("data_viagem")).between(7, 9),
        "Terceiro Trimestre"
    )
    .otherwise(
        "Quarto Trimestre"
    )
)

In [0]:
df_passagens.select(
    "id_solicitacao",
    "data_viagem",
    "tipo_despesa",
    "trimestre"
).show(20, truncate=False)

In [0]:
df_passagens.groupBy("trimestre").count().show()

In [0]:
df_map = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/referencias/centro_custo/Centro_Custo_Map.csv",
    header=True,
    inferSchema=True,
    sep=";"
)
display(df_map)

In [0]:
df_map.printSchema()
df_map.columns

In [0]:
df_passagens = (
    df_passagens.alias("p")
    .join(
        df_map.alias("m"),
        (
            (col("p.agencia") == col("m.Agencia")) &
            (col("p.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "p.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

In [0]:
df_passagens = (
    df_passagens.alias("p")
    .join(
        df_map.alias("m"),
        (
            (col("p.agencia") == col("m.Agencia")) &
            (col("p.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "p.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

In [0]:
df_passagens = df_passagens.drop(
    "centro_custo",
    "diretoria"
)

In [0]:
df_passagens.printSchema()

In [0]:
df_passagens = (
    df_passagens.alias("p")
    .join(
        df_map.alias("m"),
        (
            (col("p.agencia") == col("m.Agencia")) &
            (col("p.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "p.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

In [0]:
df_passagens.printSchema()

In [0]:
df_passagens.count()

In [0]:
df_passagens.filter(
    col("centro_custo").isNull()
).count()

In [0]:
(
    df_passagens.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("viagens_teste.silver.passagens")
)

In [0]:
df_teste_leitura = spark.table(
    "viagens_teste.silver.passagens"
)

display(df_teste_leitura)

In [0]:
print("Total de passagens:", df_passagens.count())

print(
    "Centros de custo não mapeados:",
    df_passagens.filter(
        col("centro_custo").isNull()
    ).count()
)

### Validação das justificativas

Nesta etapa verificamos se as justificativas foram carregadas corretamente e se a versão limpa foi criada sem perder o texto original.

In [0]:
display(
    df_passagens.select(
        "id_solicitacao",
        "agencia",
        "justificativa_original",
        "justificativa_limpa"
    )
)

### Preparação das justificativas para NLP

Criamos uma versão normalizada da justificativa para facilitar buscas por palavras-chave e futuras classificações. O texto original e a versão limpa continuam preservados.

In [0]:
from pyspark.sql.functions import lower, translate

df_passagens = df_passagens.withColumn(
    "justificativa_nlp",
    lower(
        translate(
            col("justificativa_limpa"),
            "áàâãäéèêëíìîïóòôõöúùûüç",
            "aaaaaeeeeiiiiooooouuuuc"
        )
    )
)

In [0]:
display(
    df_passagens.select(
        "justificativa_original",
        "justificativa_limpa",
        "justificativa_nlp"
    )
)

### Classificação inicial das justificativas

Nesta etapa aplicamos regras simples baseadas em palavras-chave para classificar as justificativas em categorias de negócio. Esta abordagem é determinística e serve como baseline antes da utilização de Machine Learning ou LLM.

In [0]:
from pyspark.sql.functions import when, col

df_passagens = df_passagens.withColumn(
    "categoria_justificativa",

    # Capacitação
    when(
        col("justificativa_nlp").rlike(
            r"\b(curso|treinamento|capacitacao|workshop)\b"
        ),
        "Capacitação"
    )

    # Eventos
    .when(
        col("justificativa_nlp").rlike(
            r"\b(congresso|seminario|evento|conferencia)\b"
        ),
        "Evento"
    )

    # Auditoria
    .when(
        col("justificativa_nlp").rlike(
            r"\b(auditoria|fiscalizacao)\b"
        ),
        "Auditoria"
    )

    # Controles / conciliação
    .when(
        col("justificativa_nlp").rlike(
            r"\b(controles financeiros|conciliacao|conciliacoes)\b"
        ),
        "Controles / Conciliação"
    )

    # Jurídico
    .when(
        col("justificativa_nlp").rlike(
            r"\b(juridico|juridica|juridicamente)\b"
        ),
        "Jurídico"
    )

    # Visita técnica
    .when(
        col("justificativa_nlp").rlike(
            r"\b(visita tecnica|vistoria)\b"
        ),
        "Visita Técnica"
    )

    # Projetos / implantação / homologação
    .when(
        col("justificativa_nlp").rlike(
            r"\b(projeto|implantacao|implementacao|homologacao|entrada em producao)\b"
        ),
        "Projeto"
    )

    # Reuniões com fornecedor
    .when(
        col("justificativa_nlp").rlike(
            r"\b(fornecedor|contratada|empresa contratada)\b"
        ),
        "Reunião com fornecedor"
    )

    # Reuniões genéricas
    .when(
        col("justificativa_nlp").rlike(
            r"\b(reuniao|alinhamento)\b"
        ),
        "Reunião"
    )

    .otherwise("Outros")
)

In [0]:
display(
    df_passagens.select(
        "id_solicitacao",
        "justificativa_limpa",
        "categoria_justificativa"
    )
)

In [0]:
display(
    df_passagens
    .groupBy("categoria_justificativa")
    .count()
    .orderBy(col("count").desc())
)

### Análise das justificativas não classificadas

Nesta etapa analisamos os registros classificados como "Outros" para identificar novos padrões, palavras-chave e possíveis melhorias nas regras de classificação.

In [0]:
display(
    df_passagens
    .filter(col("categoria_justificativa") == "Outros")
    .select(
        "id_solicitacao",
        "agencia",
        "justificativa_limpa",
        "justificativa_nlp"
    )
)

### Metadados da classificação

Nesta etapa registramos como cada justificativa foi classificada e identificamos casos que exigem revisão. Isso prepara o pipeline para futuras classificações com LLM e Machine Learning.

In [0]:
from pyspark.sql.functions import lit, when, col

df_passagens = (
    df_passagens
    .withColumn(
        "metodo_classificacao",
        lit("Regra")
    )
    .withColumn(
        "necessita_revisao",
        when(
            col("categoria_justificativa") == "Outros",
            True
        ).otherwise(False)
    )
)

In [0]:
display(
    df_passagens.select(
        "id_solicitacao",
        "justificativa_limpa",
        "categoria_justificativa",
        "metodo_classificacao",
        "necessita_revisao"
    )
)

### Tabela de regras para classificação de justificativas

As regras de classificação deixam de ficar fixas no código e passam a ser armazenadas em uma tabela Delta. Isso permite alterar categorias, padrões e prioridades sem reescrever o pipeline.

In [0]:
regras = [
    (1,  r"\b(curso|treinamento|capacitacao|workshop)\b",                  "Capacitação",              10, True),
    (2,  r"\b(congresso|seminario|evento|conferencia)\b",                  "Evento",                   20, True),
    (3,  r"\b(auditoria|fiscalizacao)\b",                                  "Auditoria",                30, True),
    (4,  r"\b(controles financeiros|conciliacao|conciliacoes)\b",          "Controles / Conciliação",  40, True),
    (5,  r"\b(juridico|juridica|juridicamente)\b",                         "Jurídico",                 50, True),
    (6,  r"\b(visita tecnica|vistoria)\b",                                 "Visita Técnica",           60, True),
    (7,  r"\b(projeto|implantacao|implementacao|homologacao|entrada em producao)\b",
                                                                             "Projeto",                  70, True),
    (8,  r"\b(fornecedor|contratada|empresa contratada)\b",                "Reunião com fornecedor",   80, True),
    (9,  r"\b(reuniao|alinhamento)\b",                                     "Reunião",                  90, True)
]

colunas = [
    "id_regra",
    "padrao_regex",
    "categoria",
    "prioridade",
    "ativo"
]

df_regras = spark.createDataFrame(regras, colunas)

In [0]:
display(
    df_regras.orderBy("prioridade")
)

In [0]:
(
    df_regras.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "viagens_teste.silver.regras_justificativas"
    )
)

In [0]:
display(
    spark.table(
        "viagens_teste.silver.regras_justificativas"
    ).orderBy("prioridade")
)

### Classificação dinâmica por tabela de regras

As regras de classificação são carregadas da tabela Delta e aplicadas dinamicamente às justificativas. A classificação anterior é mantida temporariamente para validação dos resultados.

In [0]:
from pyspark.sql import functions as F

regras_ativas = (
    spark.table("viagens_teste.silver.regras_justificativas")
    .filter(F.col("ativo") == True)
    .orderBy("prioridade")
    .collect()
)

print("Quantidade de regras ativas:", len(regras_ativas))

In [0]:
classificacao = None

for regra in regras_ativas:

    condicao = F.col("justificativa_nlp").rlike(
        regra["padrao_regex"]
    )

    if classificacao is None:
        classificacao = F.when(
            condicao,
            F.lit(regra["categoria"])
        )
    else:
        classificacao = classificacao.when(
            condicao,
            F.lit(regra["categoria"])
        )

classificacao = classificacao.otherwise("Outros")

In [0]:
df_passagens = df_passagens.withColumn(
    "categoria_tabela",
    classificacao
)

In [0]:
display(
    df_passagens.select(
        "id_solicitacao",
        "justificativa_limpa",
        "categoria_justificativa",
        "categoria_tabela"
    )
)

In [0]:
divergencias = (
    df_passagens
    .filter(
        F.col("categoria_justificativa")
        !=
        F.col("categoria_tabela")
    )
)

print(
    "Quantidade de classificações divergentes:",
    divergencias.count()
)

### Classificador oficial orientado por regras

Após validar que a classificação dinâmica produz os mesmos resultados do classificador original, a tabela `regras_justificativas` passa a ser a fonte oficial das regras de classificação.

In [0]:
df_passagens = (
    df_passagens
    .withColumn(
        "categoria_justificativa",
        F.col("categoria_tabela")
    )
    .drop("categoria_tabela")
)

In [0]:
df_passagens = (
    df_passagens
    .withColumn(
        "metodo_classificacao",
        F.lit("Regra")
    )
    .withColumn(
        "necessita_revisao",
        F.when(
            F.col("categoria_justificativa") == "Outros",
            True
        ).otherwise(False)
    )
)

In [0]:
display(
    df_passagens.select(
        "id_solicitacao",
        "justificativa_limpa",
        "categoria_justificativa",
        "metodo_classificacao",
        "necessita_revisao"
    )
)

In [0]:
display(
    df_passagens
    .groupBy(
        "categoria_justificativa",
        "metodo_classificacao",
        "necessita_revisao"
    )
    .count()
    .orderBy(F.col("count").desc())
)

### Classificação dinâmica com rastreabilidade

Além da categoria atribuída, registramos a regra responsável pela classificação. Isso permite auditar por que cada justificativa recebeu determinada categoria.

In [0]:
from pyspark.sql import functions as F

regras_ativas = (
    spark.table("viagens_teste.silver.regras_justificativas")
    .filter(F.col("ativo") == True)
    .orderBy("prioridade")
    .collect()
)

categoria_expr = None
id_regra_expr = None

for regra in regras_ativas:

    condicao = F.col("justificativa_nlp").rlike(
        regra["padrao_regex"]
    )

    if categoria_expr is None:

        categoria_expr = F.when(
            condicao,
            F.lit(regra["categoria"])
        )

        id_regra_expr = F.when(
            condicao,
            F.lit(regra["id_regra"])
        )

    else:

        categoria_expr = categoria_expr.when(
            condicao,
            F.lit(regra["categoria"])
        )

        id_regra_expr = id_regra_expr.when(
            condicao,
            F.lit(regra["id_regra"])
        )

In [0]:
categoria_expr = categoria_expr.otherwise("Outros")

id_regra_expr = id_regra_expr.otherwise(
    F.lit(None).cast("int")
)

In [0]:
df_passagens = (
    df_passagens
    .withColumn(
        "categoria_justificativa",
        categoria_expr
    )
    .withColumn(
        "id_regra_aplicada",
        id_regra_expr
    )
    .withColumn(
        "metodo_classificacao",
        F.lit("Regra")
    )
    .withColumn(
        "necessita_revisao",
        F.when(
            F.col("categoria_justificativa") == "Outros",
            True
        ).otherwise(False)
    )
)

In [0]:
display(
    df_passagens.select(
        "id_solicitacao",
        "justificativa_limpa",
        "categoria_justificativa",
        "id_regra_aplicada",
        "metodo_classificacao",
        "necessita_revisao"
    )
)

### Classificação semântica com LLM

Nesta etapa utilizamos um modelo de linguagem para interpretar semanticamente as justificativas e classificá-las nas mesmas categorias utilizadas pelo classificador baseado em regras.

Inicialmente, o LLM é utilizado como classificador independente para comparação e avaliação.

In [0]:
from pyspark.sql import functions as F

In [0]:
categorias_llm = {
    "Capacitação": "Cursos, treinamentos, workshops, certificações ou desenvolvimento profissional.",
    
    "Evento": "Congressos, seminários, conferências, fóruns ou participação em eventos institucionais.",
    
    "Auditoria": "Atividades relacionadas a auditoria, fiscalização, inspeção ou verificação formal.",
    
    "Controles / Conciliação": "Atividades relacionadas a controles financeiros, conciliações, conferências ou revisões financeiras.",
    
    "Jurídico": "Demandas judiciais, jurídicas, legais ou acompanhamento jurídico.",
    
    "Visita Técnica": "Visitas técnicas, vistorias ou acompanhamento técnico presencial.",
    
    "Projeto": "Projetos, implantação, implementação, homologação, entrada em produção ou acompanhamento de entregas.",
    
    "Reunião com fornecedor": "Reuniões, alinhamentos ou negociações presenciais envolvendo fornecedores ou empresas contratadas.",
    
    "Reunião": "Reuniões, alinhamentos ou encontros presenciais que não se enquadram nas demais categorias.",
    
    "Outros": "Justificativas que não se enquadram claramente em nenhuma das categorias anteriores."
}

In [0]:
df_teste_llm = (
    df_passagens
    .select(
        "id_solicitacao",
        "justificativa_limpa",
        "categoria_justificativa"
    )
    .limit(5)
    .withColumn(
        "categoria_llm",
        F.ai_classify(
            "justificativa_limpa",
            categorias_llm
        )
    )
)

### Teste de classificação com LLM via SQL

Como o wrapper PySpark `ai_classify` não está disponível neste ambiente, utilizamos diretamente a função SQL do Databricks para executar a classificação semântica.

In [0]:
df_passagens.select(
    "id_solicitacao",
    "justificativa_limpa",
    "categoria_justificativa"
).limit(5).createOrReplaceTempView("teste_llm")

In [0]:
df_teste_llm = spark.sql("""
SELECT
    id_solicitacao,
    justificativa_limpa,
    categoria_justificativa,

    ai_classify(
        justificativa_limpa,

        '{
            "Capacitação": "Cursos, treinamentos, workshops, certificações ou desenvolvimento profissional.",
            "Evento": "Congressos, seminários, conferências, fóruns ou eventos institucionais.",
            "Auditoria": "Auditoria, fiscalização, inspeção ou verificação formal.",
            "Controles / Conciliação": "Controles financeiros, conciliações, conferências ou revisões financeiras.",
            "Jurídico": "Demandas judiciais, jurídicas, legais ou acompanhamento jurídico.",
            "Visita Técnica": "Visitas técnicas, vistorias ou acompanhamento técnico presencial.",
            "Projeto": "Projetos, implantação, implementação, homologação ou entrada em produção.",
            "Reunião com fornecedor": "Reuniões ou alinhamentos envolvendo fornecedores ou empresas contratadas.",
            "Reunião": "Reuniões ou alinhamentos que não se enquadram nas demais categorias.",
            "Outros": "Casos que não se enquadram claramente nas categorias anteriores."
        }',

        MAP(
            'version', '2.1',
            'enableConfidenceScores', 'true',
            'enableRationales', 'true',
            'instructions',
            'Classifique o motivo principal da viagem corporativa em apenas uma categoria.'
        )
    ) AS resultado_llm

FROM teste_llm
""")

In [0]:
display(df_teste_llm)

### Extração dos resultados do LLM

O retorno do modelo é separado em categoria, confiança e justificativa da classificação para permitir comparação com o classificador baseado em regras.

In [0]:
df_teste_llm_v2 = spark.sql("""
WITH classificado AS (
    SELECT
        id_solicitacao,
        justificativa_limpa,
        categoria_justificativa,

        ai_classify(
            justificativa_limpa,

            '{
                "Capacitação": "Cursos, treinamentos, workshops, certificações ou desenvolvimento profissional.",
                "Evento": "Congressos, seminários, conferências, fóruns ou eventos institucionais.",
                "Auditoria": "Auditoria, fiscalização, inspeção ou verificação formal.",
                "Controles / Conciliação": "Controles financeiros, conciliações, conferências ou revisões financeiras.",
                "Jurídico": "Demandas judiciais, jurídicas, legais ou acompanhamento jurídico.",
                "Visita Técnica": "Visitas técnicas, vistorias ou acompanhamento técnico presencial.",
                "Projeto": "Projetos, implantação, implementação, homologação ou entrada em produção.",
                "Reunião com fornecedor": "Reuniões ou alinhamentos envolvendo fornecedores ou empresas contratadas.",
                "Reunião": "Reuniões ou alinhamentos que não se enquadram nas demais categorias.",
                "Outros": "Casos que não se enquadram claramente nas categorias anteriores."
            }',

            MAP(
                'version', '2.1',
                'enableConfidenceScores', 'true',
                'enableRationales', 'true',
                'instructions',
                'Classifique o motivo principal da viagem corporativa em apenas uma categoria.'
            )
        ) AS resultado_llm

    FROM teste_llm
)

SELECT
    id_solicitacao,
    justificativa_limpa,
    categoria_justificativa,

    try_variant_get(
        resultado_llm,
        '$.response[0].value',
        'string'
    ) AS categoria_llm,

    try_variant_get(
        resultado_llm,
        '$.response[0].confidence_score',
        'double'
    ) AS confianca_llm,

    try_variant_get(
        resultado_llm,
        '$.response[0].rationale',
        'string'
    ) AS racional_llm

FROM classificado
""")

In [0]:
display(df_teste_llm_v2)

In [0]:
from pyspark.sql import functions as F

df_teste_llm_v2 = df_teste_llm_v2.withColumn(
    "regra_llm_concordam",
    F.col("categoria_justificativa") == F.col("categoria_llm")
).withColumn(
    "necessita_revisao_llm",
    F.when(
        (F.col("categoria_justificativa") != F.col("categoria_llm")) |
        (F.col("confianca_llm") < 0.75),
        True
    ).otherwise(False)
)

In [0]:
display(
    df_teste_llm_v2.select(
        "id_solicitacao",
        "categoria_justificativa",
        "categoria_llm",
        "confianca_llm",
        "regra_llm_concordam",
        "necessita_revisao_llm",
        "racional_llm"
    )
)

### Avaliação do classificador LLM em todas as passagens

Executamos o classificador semântico nas justificativas e materializamos os resultados para comparar a classificação por regras com a classificação do LLM, evitando novas chamadas ao modelo durante as análises.

In [0]:
df_passagens.select(
    "id_solicitacao",
    "agencia",
    "justificativa_limpa",
    "categoria_justificativa",
    "id_regra_aplicada"
).createOrReplaceTempView("passagens_para_llm")

In [0]:
df_llm_passagens = spark.sql("""
WITH classificado AS (

    SELECT
        id_solicitacao,
        agencia,
        justificativa_limpa,
        categoria_justificativa,
        id_regra_aplicada,

        ai_classify(
            justificativa_limpa,

            '{
                "Capacitação": "Cursos, treinamentos, workshops, certificações ou desenvolvimento profissional.",
                "Evento": "Congressos, seminários, conferências, fóruns ou eventos institucionais.",
                "Auditoria": "Auditoria, fiscalização, inspeção ou verificação formal.",
                "Controles / Conciliação": "Controles financeiros, conciliações, conferências ou revisões financeiras.",
                "Jurídico": "Demandas judiciais, jurídicas, legais ou acompanhamento jurídico.",
                "Visita Técnica": "Visitas técnicas, vistorias ou acompanhamento técnico presencial.",
                "Projeto": "Projetos, implantação, implementação, homologação ou entrada em produção.",
                "Reunião com fornecedor": "Reuniões ou alinhamentos envolvendo fornecedores ou empresas contratadas.",
                "Reunião": "Reuniões ou alinhamentos que não se enquadram nas demais categorias.",
                "Outros": "Casos que não se enquadram claramente nas categorias anteriores."
            }',

            MAP(
                'version', '2.1',
                'enableConfidenceScores', 'true',
                'enableRationales', 'true',
                'instructions',
                'Classifique o motivo principal da viagem corporativa em apenas uma categoria.'
            )
        ) AS resultado_llm

    FROM passagens_para_llm
)

SELECT
    id_solicitacao,
    agencia,
    justificativa_limpa,
    categoria_justificativa,
    id_regra_aplicada,

    try_variant_get(
        resultado_llm,
        '$.response[0].value',
        'string'
    ) AS categoria_llm,

    try_variant_get(
        resultado_llm,
        '$.response[0].confidence_score',
        'double'
    ) AS confianca_llm,

    try_variant_get(
        resultado_llm,
        '$.response[0].rationale',
        'string'
    ) AS racional_llm

FROM classificado
""")

In [0]:
(
    df_llm_passagens.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "viagens_teste.silver.classificacao_llm_passagens"
    )
)

In [0]:
df_llm_passagens = spark.table(
    "viagens_teste.silver.classificacao_llm_passagens"
)

In [0]:
from pyspark.sql import functions as F

df_llm_passagens = (
    df_llm_passagens

    .withColumn(
        "regra_llm_concordam",
        F.col("categoria_justificativa") ==
        F.col("categoria_llm")
    )

    .withColumn(
        "necessita_revisao_llm",
        F.when(
            (F.col("categoria_justificativa") != F.col("categoria_llm")) |
            (F.col("confianca_llm") < 0.75) |
            (F.col("categoria_llm").isNull()),
            True
        ).otherwise(False)
    )
)

In [0]:
display(
    df_llm_passagens
    .groupBy("regra_llm_concordam")
    .count()
)

In [0]:
display(
    df_llm_passagens
    .groupBy("necessita_revisao_llm")
    .count()
)

In [0]:
display(
    df_llm_passagens
    .filter(
        F.col("necessita_revisao_llm") == True
    )
    .select(
        "id_solicitacao",
        "justificativa_limpa",
        "categoria_justificativa",
        "categoria_llm",
        "confianca_llm",
        "racional_llm"
    )
    .orderBy("confianca_llm")
)

### Identificação do motivo de revisão

Os registros sinalizados são separados pelo motivo da revisão, permitindo distinguir divergências entre classificadores, baixa confiança do LLM e possíveis falhas de resposta.

In [0]:
from pyspark.sql import functions as F

df_llm_passagens = df_llm_passagens.withColumn(
    "motivo_revisao",
    
    F.when(
        F.col("categoria_llm").isNull(),
        "LLM sem resposta"
    )
    
    .when(
        F.col("categoria_justificativa") != F.col("categoria_llm"),
        "Divergência Regra x LLM"
    )
    
    .when(
        F.col("confianca_llm") < 0.75,
        "Baixa confiança LLM"
    )
    
    .otherwise("Sem revisão")
)

In [0]:
display(
    df_llm_passagens
    .groupBy("motivo_revisao")
    .count()
    .orderBy(F.col("count").desc())
)

In [0]:
display(
    df_llm_passagens
    .filter(F.col("necessita_revisao_llm") == True)
    .select(
        "id_solicitacao",
        "justificativa_limpa",
        "categoria_justificativa",
        "id_regra_aplicada",
        "categoria_llm",
        "confianca_llm",
        "motivo_revisao",
        "racional_llm"
    )
    .orderBy(
        "motivo_revisao",
        "confianca_llm"
    )
)

### Classificação em duas dimensões

A justificativa passa a ser analisada em duas dimensões independentes:

- Tipo de atividade: o que foi realizado durante a viagem.
- Tema da viagem: qual assunto ou área motivou a atividade.

Essa separação reduz ambiguidades entre categorias como reunião, projeto, fornecedor e controles financeiros.

In [0]:
from pyspark.sql import functions as F

df_passagens = df_passagens.withColumn(
    "tipo_atividade",

    F.when(
        F.col("justificativa_nlp").rlike(
            r"\b(curso|treinamento|capacitacao|workshop)\b"
        ),
        "Capacitação"
    )

    .when(
        F.col("justificativa_nlp").rlike(
            r"\b(congresso|seminario|evento|conferencia)\b"
        ),
        "Evento"
    )

    .when(
        F.col("justificativa_nlp").rlike(
            r"\b(visita|vistoria)\b"
        ),
        "Visita / Vistoria"
    )

    .when(
        F.col("justificativa_nlp").rlike(
            r"\b(reuniao|alinhamento)\b"
        ),
        "Reunião"
    )

    .when(
        F.col("justificativa_nlp").rlike(
            r"\b(acompanhamento)\b"
        ),
        "Acompanhamento"
    )

    .when(
        F.col("justificativa_nlp").rlike(
            r"\b(apoio presencial)\b"
        ),
        "Apoio Presencial"
    )

    .otherwise("Outros")
)

In [0]:
df_passagens = df_passagens.withColumn(
    "tema_justificativa",

    F.when(
        F.col("justificativa_nlp").rlike(
            r"\b(fornecedor|fornecedores|contratada|contratadas|contrato|contratos|contratual|contratuais)\b"
        ),
        "Contratos / Fornecedores"
    )

    .when(
        F.col("justificativa_nlp").rlike(
            r"\b(financeiro|financeira|financeiras|conciliacao|conciliacoes|projecoes|premissas|controles financeiros)\b"
        ),
        "Financeiro / Controles"
    )

    .when(
        F.col("justificativa_nlp").rlike(
            r"\b(projeto|implantacao|implementacao|homologacao|sistema|sistemas|tecnologia|integracao|funcionalidade)\b"
        ),
        "Tecnologia / Projetos"
    )

    .when(
        F.col("justificativa_nlp").rlike(
            r"\b(juridico|juridica|legal)\b"
        ),
        "Jurídico"
    )

    .when(
        F.col("justificativa_nlp").rlike(
            r"\b(gestao de pessoas|planejamento de pessoas|desenvolvimento organizacional)\b"
        ),
        "Pessoas"
    )

    .when(
        F.col("justificativa_nlp").rlike(
            r"\b(comunicacao|campanha institucional|institucional)\b"
        ),
        "Comunicação / Institucional"
    )

    .when(
        F.col("justificativa_nlp").rlike(
            r"\b(auditoria|fiscalizacao)\b"
        ),
        "Auditoria / Fiscalização"
    )

    .otherwise("Outros")
)

In [0]:
display(
    df_passagens.select(
        "id_solicitacao",
        "justificativa_limpa",
        "tipo_atividade",
        "tema_justificativa"
    )
)

### Classificação com LLM em duas dimensões

O LLM passa a interpretar cada justificativa em duas dimensões independentes:

- Tipo de atividade realizada;
- Tema ou assunto principal da viagem.

Isso reduz ambiguidades entre conceitos como reunião, projeto, fornecedor e controles financeiros.

In [0]:
df_passagens.select(
    "id_solicitacao",
    "justificativa_limpa",
    "tipo_atividade",
    "tema_justificativa"
).limit(5).createOrReplaceTempView("teste_llm_duplo")

In [0]:
df_llm_duplo = spark.sql("""
WITH classificado AS (

    SELECT
        id_solicitacao,
        justificativa_limpa,
        tipo_atividade,
        tema_justificativa,

        ai_classify(
            justificativa_limpa,

            '{
                "Reunião": "Reuniões, alinhamentos, encontros ou discussões presenciais.",
                "Visita / Vistoria": "Visitas presenciais, visitas técnicas, inspeções ou vistorias.",
                "Capacitação": "Cursos, treinamentos, workshops ou atividades de desenvolvimento profissional.",
                "Evento": "Congressos, seminários, conferências ou eventos institucionais.",
                "Acompanhamento": "Atividades de acompanhamento presencial de execução, implantação ou operação.",
                "Apoio Presencial": "Deslocamento para prestar apoio presencial a uma equipe ou atividade.",
                "Outros": "Atividades que não se enquadram claramente nas categorias anteriores."
            }',

            MAP(
                'version', '2.1',
                'enableConfidenceScores', 'true',
                'enableRationales', 'true',
                'instructions',
                'Classifique apenas o tipo de atividade realizada durante a viagem.'
            )
        ) AS resultado_tipo,

        ai_classify(
            justificativa_limpa,

            '{
                "Contratos / Fornecedores": "Assuntos relacionados a contratos, fornecedores, empresas contratadas ou entregas contratuais.",
                "Financeiro / Controles": "Planejamento financeiro, projeções, conciliações, controles ou análises financeiras.",
                "Tecnologia / Projetos": "Projetos, sistemas, tecnologia, implantação, homologação, integração ou desenvolvimento de soluções.",
                "Jurídico": "Demandas jurídicas, legais ou judiciais.",
                "Pessoas": "Gestão de pessoas, desenvolvimento organizacional, planejamento de pessoas ou capacitação da equipe.",
                "Comunicação / Institucional": "Comunicação, campanhas, ações institucionais ou divulgação.",
                "Auditoria / Fiscalização": "Auditoria, fiscalização, inspeção formal ou controle.",
                "Outros": "Temas que não se enquadram claramente nas categorias anteriores."
            }',

            MAP(
                'version', '2.1',
                'enableConfidenceScores', 'true',
                'enableRationales', 'true',
                'instructions',
                'Classifique apenas o tema principal ou assunto da viagem.'
            )
        ) AS resultado_tema

    FROM teste_llm_duplo
)

SELECT
    id_solicitacao,
    justificativa_limpa,

    tipo_atividade AS tipo_regra,

    try_variant_get(
        resultado_tipo,
        '$.response[0].value',
        'string'
    ) AS tipo_llm,

    try_variant_get(
        resultado_tipo,
        '$.response[0].confidence_score',
        'double'
    ) AS confianca_tipo_llm,

    tema_justificativa AS tema_regra,

    try_variant_get(
        resultado_tema,
        '$.response[0].value',
        'string'
    ) AS tema_llm,

    try_variant_get(
        resultado_tema,
        '$.response[0].confidence_score',
        'double'
    ) AS confianca_tema_llm

FROM classificado
""")

In [0]:
display(df_llm_duplo)

In [0]:
display(
    df_passagens
    .filter(F.col("id_solicitacao") == "VN-1005")
    .select(
        "justificativa_limpa",
        "tipo_atividade",
        "tema_justificativa"
    )
)

In [0]:
df_passagens.select(
    "id_solicitacao",
    "agencia",
    "justificativa_limpa",
    "tipo_atividade",
    "tema_justificativa"
).createOrReplaceTempView("passagens_llm_duplo")

In [0]:
df_llm_duplo_36 = spark.sql("""
WITH classificado AS (

    SELECT
        id_solicitacao,
        agencia,
        justificativa_limpa,
        tipo_atividade,
        tema_justificativa,

        ai_classify(
            justificativa_limpa,
            '{
                "Reunião": "Reuniões, alinhamentos, encontros ou discussões presenciais.",
                "Visita / Vistoria": "Visitas presenciais, visitas técnicas, inspeções ou vistorias.",
                "Capacitação": "Cursos, treinamentos, workshops ou atividades de desenvolvimento profissional.",
                "Evento": "Congressos, seminários, conferências ou eventos institucionais.",
                "Acompanhamento": "Atividades de acompanhamento presencial de execução, implantação ou operação.",
                "Apoio Presencial": "Deslocamento para prestar apoio presencial a uma equipe ou atividade.",
                "Outros": "Atividades que não se enquadram claramente nas categorias anteriores."
            }',
            MAP(
                'version', '2.1',
                'enableConfidenceScores', 'true',
                'enableRationales', 'true',
                'instructions',
                'Classifique apenas o tipo de atividade realizada durante a viagem.'
            )
        ) AS resultado_tipo,

        ai_classify(
            justificativa_limpa,
            '{
                "Contratos / Fornecedores": "Assuntos relacionados a contratos, fornecedores, empresas contratadas ou entregas contratuais.",
                "Financeiro / Controles": "Planejamento financeiro, projeções, conciliações, controles ou análises financeiras.",
                "Tecnologia / Projetos": "Projetos, sistemas, tecnologia, implantação, homologação, integração ou desenvolvimento de soluções.",
                "Jurídico": "Demandas jurídicas, legais ou judiciais.",
                "Pessoas": "Gestão de pessoas, desenvolvimento organizacional, planejamento de pessoas ou capacitação da equipe.",
                "Comunicação / Institucional": "Comunicação, campanhas, ações institucionais ou divulgação.",
                "Auditoria / Fiscalização": "Auditoria, fiscalização, inspeção formal ou controle.",
                "Outros": "Temas que não se enquadram claramente nas categorias anteriores."
            }',
            MAP(
                'version', '2.1',
                'enableConfidenceScores', 'true',
                'enableRationales', 'true',
                'instructions',
                'Classifique apenas o tema principal ou assunto da viagem.'
            )
        ) AS resultado_tema

    FROM passagens_llm_duplo
)

SELECT
    id_solicitacao,
    agencia,
    justificativa_limpa,

    tipo_atividade AS tipo_regra,

    try_variant_get(
        resultado_tipo,
        '$.response[0].value',
        'string'
    ) AS tipo_llm,

    try_variant_get(
        resultado_tipo,
        '$.response[0].confidence_score',
        'double'
    ) AS confianca_tipo_llm,

    tema_justificativa AS tema_regra,

    try_variant_get(
        resultado_tema,
        '$.response[0].value',
        'string'
    ) AS tema_llm,

    try_variant_get(
        resultado_tema,
        '$.response[0].confidence_score',
        'double'
    ) AS confianca_tema_llm

FROM classificado
""")

In [0]:
(
    df_llm_duplo_36.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "viagens_teste.silver.classificacao_llm_dupla_passagens"
    )
)

In [0]:
df_llm_duplo_36 = spark.table(
    "viagens_teste.silver.classificacao_llm_dupla_passagens"
)

In [0]:
from pyspark.sql import functions as F

df_llm_duplo_36 = (
    df_llm_duplo_36
    .withColumn(
        "tipo_concorda",
        F.col("tipo_regra") == F.col("tipo_llm")
    )
    .withColumn(
        "tema_concorda",
        F.col("tema_regra") == F.col("tema_llm")
    )
)

In [0]:
display(
    df_llm_duplo_36
    .groupBy("tipo_concorda")
    .count()
)

In [0]:
display(
    df_llm_duplo_36
    .groupBy("tema_concorda")
    .count()
)

### Análise das divergências entre regras e LLM

Nesta etapa analisamos apenas os registros em que o classificador baseado em regras e o LLM produziram resultados diferentes.

In [0]:
display(
    df_llm_duplo_36
    .filter(
        (F.col("tipo_concorda") == False) |
        (F.col("tema_concorda") == False)
    )
    .select(
        "id_solicitacao",
        "justificativa_limpa",

        "tipo_regra",
        "tipo_llm",
        "confianca_tipo_llm",

        "tema_regra",
        "tema_llm",
        "confianca_tema_llm"
    )
)

### Revisão humana das divergências

Os casos em que os classificadores produziram respostas diferentes são revisados manualmente. A decisão humana é armazenada separadamente para preservar a rastreabilidade entre regras, LLM e classificação final.

In [0]:
revisoes_humanas = [
    (
        "AT-5003",
        "Acompanhamento",
        "Jurídico",
        "Atividade principal descrita como acompanhamento jurídico; reunião é complementar."
    ),
    (
        "AT-5004",
        "Capacitação",
        "Pessoas",
        "Treinamento corporativo voltado ao desenvolvimento de lideranças."
    ),
    (
        "AT-5017",
        "Apoio Presencial",
        "Comunicação / Institucional",
        "A atividade principal é apoio presencial à equipe durante um evento."
    ),
    (
        "VN-1008",
        "Reunião",
        "Contratos / Fornecedores",
        "A reunião trata de processo de contratação e requisitos."
    ),
    (
        "VN-1016",
        "Reunião",
        "Pessoas",
        "A atividade realizada foi uma reunião; capacitação é apenas parte do assunto tratado."
    ),
    (
        "VN-1018",
        "Reunião",
        "Outros",
        "O texto fala em gestão e indicadores, mas não há evidência suficiente para classificar como Pessoas."
    )
]

colunas_revisao = [
    "id_solicitacao",
    "tipo_atividade_humano",
    "tema_justificativa_humano",
    "justificativa_revisao"
]

df_revisao_humana = spark.createDataFrame(
    revisoes_humanas,
    colunas_revisao
)

display(df_revisao_humana)

In [0]:
(
    df_revisao_humana.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "viagens_teste.silver.revisao_humana_justificativas"
    )
)

In [0]:
df_classificacao_final = (
    df_llm_duplo_36
    .join(
        df_revisao_humana,
        on="id_solicitacao",
        how="left"
    )
)

In [0]:
from pyspark.sql import functions as F

df_classificacao_final = (
    df_classificacao_final

    .withColumn(
        "tipo_atividade_final",
        F.coalesce(
            F.col("tipo_atividade_humano"),
            F.col("tipo_regra")
        )
    )

    .withColumn(
        "tema_justificativa_final",
        F.coalesce(
            F.col("tema_justificativa_humano"),
            F.col("tema_regra")
        )
    )

    .withColumn(
        "metodo_decisao_final",
        F.when(
            F.col("tipo_atividade_humano").isNotNull() |
            F.col("tema_justificativa_humano").isNotNull(),
            "Revisão Humana"
        ).otherwise(
            "Concordância Regra x LLM"
        )
    )
)

In [0]:
display(
    df_classificacao_final.select(
        "id_solicitacao",
        "justificativa_limpa",

        "tipo_regra",
        "tipo_llm",
        "tipo_atividade_final",

        "tema_regra",
        "tema_llm",
        "tema_justificativa_final",

        "metodo_decisao_final"
    )
)

### Incorporação dos rótulos finais à Silver

Os resultados validados pelo fluxo de regras, LLM e revisão humana são incorporados à tabela principal de passagens, preservando a classificação final e o método utilizado na decisão.

In [0]:
df_rotulos_finais = (
    df_classificacao_final
    .select(
        "id_solicitacao",
        "tipo_atividade_final",
        "tema_justificativa_final",
        "metodo_decisao_final"
    )
)

In [0]:
colunas_finais = [
    "tipo_atividade_final",
    "tema_justificativa_final",
    "metodo_decisao_final"
]

for coluna in colunas_finais:
    if coluna in df_passagens.columns:
        df_passagens = df_passagens.drop(coluna)

In [0]:
df_passagens = (
    df_passagens
    .join(
        df_rotulos_finais,
        on="id_solicitacao",
        how="left"
    )
)

In [0]:
print("Total de passagens:", df_passagens.count())

print(
    "Rótulos finais ausentes:",
    df_passagens
    .filter(
        F.col("tipo_atividade_final").isNull() |
        F.col("tema_justificativa_final").isNull()
    )
    .count()
)

In [0]:
display(
    df_passagens.select(
        "id_solicitacao",
        "justificativa_limpa",
        "tipo_atividade",
        "tipo_atividade_final",
        "tema_justificativa",
        "tema_justificativa_final",
        "metodo_decisao_final"
    )
)

In [0]:
display(
    df_passagens
    .groupBy("metodo_decisao_final")
    .count()
)

In [0]:
(
    df_passagens.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "viagens_teste.silver.passagens"
    )
)